In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np
import hashlib

root = Path("/home/ibm-user/shared")
v2 = root / "guardian_v2_outputs"

paths = {
    "data": root / "clean_transactions.csv",
    "risk": root / "account_risk.csv",
    "guardian": v2 / "freeze_plan_candidate.csv",
    "granite": v2 / "freeze_plan_explained.csv",
    "manifest": v2 / "run_manifest.json"
}

print("=== GOLDEN HOUR V2 INTEGRATION ===")

for name, path in paths.items():
    assert path.is_file(), f"Missing {name}: {path}"
    print(f"PASS | {name} found")

risk = pd.read_csv(paths["risk"], dtype={"acct": str})
guardian = pd.read_csv(paths["guardian"], dtype={"acct": str})
granite = pd.read_csv(paths["granite"], dtype={"acct": str})

# Check account IDs
assert risk["acct"].is_unique
assert guardian["acct"].is_unique
assert granite["acct"].is_unique

assert set(guardian["acct"]).issubset(set(risk["acct"]))
assert set(guardian["acct"]) == set(granite["acct"])
print("PASS | Account IDs match")

# Confirm Guardian uses detector risk scores
comparison = guardian[["acct", "risk_score"]].merge(
    risk[["acct", "risk_score"]],
    on="acct",
    validate="one_to_one",
    suffixes=("_guardian", "_alex")
)

assert np.allclose(
    comparison["risk_score_guardian"],
    comparison["risk_score_alex"],
    rtol=0,
    atol=1e-7
)
print("PASS | Risk scores match")

# Ensure the explanatory output retains V2's columns
pd.testing.assert_frame_equal(
    guardian.sort_values("acct").reset_index(drop=True),
    granite[guardian.columns].sort_values("acct").reset_index(drop=True),
    check_exact=False,
    check_dtype=False,
    rtol=1e-12,
    atol=1e-8
)
print("PASS | Guardian data preserved")

# Check the explanations
assert len(guardian) == len(granite) == 10
assert granite["explanation"].notna().all()
assert granite["explanation"].str.strip().ne("").all()
assert granite["explanation"].str.contains(
    "human approval", case=False, regex=False
).all()

print("PASS | 10 Granite explanations")
print("PASS | Human approval included")

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

risk_hash = sha256(paths["risk"])
print("\nDetector SHA-256:", risk_hash)

expected = (
    "b9060d54b71da0fb8592a29c5c39ca9788f77dabfffd21051f3a0a9c04a7e0ad"
)

assert risk_hash == expected, "Risk input hash changed!"
print("PASS | Verified risk-file hash")

print("\nV2 HANDOFF INTEGRATION PASSED")
